# 3장 — 온톨로지로 첫 번째 지식 그래프 만들기

**책**: Alessandro Negro 외, *Knowledge Graphs and LLMs in Action* (Manning, 2025) 3장

**교재 저장소 폴더**: [`src/ch03`](https://github.com/restful3/ds4th_study/tree/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/src/ch03) — MEAP 번호라 책 3장 = 저장소 `ch03`, 리스팅 번호는 **책 = 파일 − 3**

**해설 원본**: [`03_..._ko_explained.md`](https://github.com/restful3/ds4th_study/blob/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/03_create_your_first_knowledge_graph_from_ontologies_ko_explained.md)

**원서 저장소**: [alenegro81/knowledge-graphs-and-llms-in-action `chapters/ch03`](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/tree/main/chapters/ch03)

---

이 노트북 하나로 3장을 이해하고 직접 돌려볼 수 있도록 만들었다.

| 절 | 내용 | 실행 |
| --- | --- | --- |
| 3.1 | 문제·도메인·데이터 이해 (임상의, HPO) | 실습 3-A |
| 3.2 | RDF vs LPG — 왜 LPG인가 | 실습 3-B |
| 3.3 | KG 구축 파이프라인 (책 3.13\~3.24) | 리스팅 실행 |
| 3.4 | 임상의의 진단 질의 (책 3.25\~3.26) | 리스팅 실행 |
| 3.5 | 온톨로지 기반 추론 (책 3.27\~3.28) | 리스팅 실행 |
| — | 요약·용어·연습문제 | 실습 3-C |

> **한 문장 요약** — 표준 어휘인 **온톨로지(HPO)** 를 축으로 삼아, RDF/XML 온톨로지 파일과 TSV 주석 파일이라는 **형식이 다른 두 소스** 를 하나의 LPG 지식 그래프로 합치고, 그 위에서 질의와 추론으로 **희귀 질환 진단** 을 돕는다.

### 이 장의 멘탈 모델

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/4059f0d9a9ad651fde89b6f3cb0fe563f3575fb0e1425f2ec44a3d8deec0d03f.jpg" width="820" alt="지식 그래프 구축 과정의 멘탈 모델">

*그림 3.1 — 지식 그래프 구축 과정을 CRISP-DM 모델의 구체화로 나타낸 멘탈 모델. 가운데는 이 예제에서 KG를 만드는 단계들, 아래쪽은 다른 시나리오에도 재사용할 수 있는 추상적 파이프라인이다. 비즈니스 목표 이해에서 시작해 임상의의 활동을 지원하는 KG 질의 정의까지 이른다.*

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/bacd29bfa050debe20ad1e1a84b4e4c5d5cab28724112aaf41a3e3738e13fc76.jpg" width="820" alt="KG에 맞게 조정한 CRISP-DM 모델">

*그림 3.2 — 지식 그래프에 맞게 조정한 CRISP-DM 모델. **비즈니스 이해 · 데이터 이해 · 데이터 준비 · KG 모델 생성/갱신** 이 이 장에서 다루는 핵심 단계다. 이 노트북의 절 구성이 그대로 이 그림을 따른다.*

## 이 장이 푸는 문제 — 임상의의 희귀 질환 진단

**타깃 페르소나는 임상의(clinician)** 다. 임상의의 가장 까다로운 업무는 환자에게서 관찰한 증상, 즉 **표현형 특징(phenotypic traits)** 을 근거로 질병을 정확히 식별하는 일이고, 희귀 증후군일 때 특히 어렵다.

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/66f9aea0decdb71b9b32e6f3fa5229458afc3f66fc3fc8301df963cb1c76b370.jpg" width="820" alt="비즈니스 도메인 이해 단계">

*그림 3.3 — 임상의 활동을 지원하는 KG를 만들기 위해 비즈니스 도메인을 이해하는 단계. 기술적 측면과 엄밀히 연관되지는 않지만 다음 단계들의 근본적인 토대가 된다.*

지식 베이스는 두 기능을 갖춰야 한다.

- **표현형 도메인의 맥락적 서술** — 같은 장기·계통에 속한 표현형 이상들이 서로 명시적으로 연결되어야 한다

- **표현형 이상과 질병 사이 관계의 서술** — 임상의가 그 연결의 **출처(source)** 까지 추적할 수 있어야 한다

두 번째 요구가 이 장의 기술 선택을 결정한다. "이 질병과 이 증상이 연관된다"는 사실만으로는 부족하고, **어느 논문(PMID)에서, 어떤 증거 수준으로, 누가 언제 주석했는지** 가 관계마다 달라붙어야 한다.

### 회색 지대 — 제1형 당뇨병은 질병인가 증상인가

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/33ae8039550d3f5a594db1a4b6c1e38bc924a3f89843abc47ee78728e3e9f521.jpg" width="760" alt="제1형 당뇨병의 두 가지 분류">

*그림 3.4 — 제1형 당뇨병(Type 1 diabetes mellitus)은 질병으로 볼 수도, 표현형적 특징으로 볼 수도 있다. 맥락에 따라 서로 다른 두 개의 ID를 채택한다.*

임상의의 일에는 회색 지대가 있다. 당뇨병은 하나의 질병으로 분류될 수도 있고, 다른 희귀 증후군의 표현형적 특징으로 분류될 수도 있다. 그래서 두 ID를 함께 쓴다.

- `OMIM:222100` — **질병** 으로서의 제1형 당뇨병

- `HP:0100651` — **표현형 특징** 으로서의 제1형 당뇨병

이 노트북 3.4절의 진단 시나리오가 바로 이 이중성을 활용한다.

## 학습 목표

해설판이 밝힌 이 장의 목표는 세 가지다.

1. 사용 사례에 근거해 **가장 알맞은 지식 그래프 기술을 고르는 법**

2. 임상의의 활동을 지원하는 **지식 그래프를 실제로 구성하는 법**

3. 지식 그래프 위에서 **분석과 온톨로지 기반 추론(ontology-based reasoning)을 수행하는 법**

### 왜 KG 구축이 복잡한가

통합해야 할 소스가 여러 층위에서 제각각이기 때문이다. 형식이 다르고(XML·CSV·JSON), 저장 기술이 다르고(관계형·문서지향), 문법이 다르고(`2022-08-09` vs `9 August 2022`), 무엇보다 **데이터가 뜻하는 의미** 가 다르다. 의료 도메인의 전형적 함정:

| 문제 | 예시 |
| --- | --- |
| 같은 개념, 여러 표현 | 제2형 당뇨병 vs 케토시스 저항성 당뇨병 |
| 같은 약어, 다른 개념 | PE = 신체검사(physical examination) 또는 폐색전증(pulmonary embolism) |
| 세분성 차이 | necrosis(괴사) vs lobular necrosis(소엽성 괴사) |

해법이 **의미 통합(semantic integration)** 이다. 하나 이상의 **온톨로지** 를 들어오는 데이터의 **기준 스키마이자 어휘** 로 채택하면, 온톨로지가 이질적 정보 사이의 **중개자** 가 된다. 소스의 로컬 스키마를 온톨로지의 기준 스키마에 잇는 대응 관계가 **매핑(mapping)** 이다.

---
## 실행 환경

이 노트북은 교재 루트의 `.venv`(Python 3.10) 커널에서 돌아간다. 처음이면 교재 루트에서 [`setup_env.py`](https://github.com/restful3/ds4th_study/blob/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/setup_env.py) 를 한 번 실행한다.

```bash
python3 setup_env.py          # .venv, util/, config.ini, data/, dataset/ 준비
```

Neo4j는 Docker로 띄운다. 3장은 **Neosemantics(n10s)** 플러그인이 반드시 필요하고, 4장에서 쓸 **GDS** 도 함께 넣어 둔다.

```bash
docker run -d --name kglm-neo4j \
  -p 127.0.0.1:7474:7474 -p 127.0.0.1:7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -e 'NEO4J_PLUGINS=["graph-data-science","n10s","apoc"]' \
  -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*,n10s.*' \
  -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*,n10s.*' \
  -v kglm-neo4j-data:/data neo4j:5.26
```

> **Community 에디션 제약** — 책은 Neo4j **Enterprise** 5.20 에서 테스트했고, 책 3.13은 `CREATE DATABASE hpo` 로 전용 데이터베이스를 만든다. Community 에디션은 멀티 데이터베이스를 지원하지 않아 이 명령이 `UnsupportedAdministrationCommand` 로 실패한다. 그래서 이 노트북은 기본 `neo4j` 데이터베이스에 그래프를 만든다. 학습 내용에는 영향이 없다.

In [1]:
"""환경 점검 — 이 셀이 통과하면 이후 모든 셀을 돌릴 수 있다."""
from kgbook import cypher, BOOK_ROOT

DB = "neo4j"          # Community 에디션이라 기본 DB 사용 (책 3.13의 hpo DB 대신)
HERE = BOOK_ROOT / "chapter_03_create_your_first_knowledge_graph_from_ontologies/src/ch03"

print("교재 루트 :", BOOK_ROOT.name)
print("config.ini:", cypher.neo4j_params())

with cypher.driver() as drv, drv.session(database=DB) as s:
    info = s.run(
        "CALL dbms.components() YIELD name, versions, edition "
        "RETURN versions[0] AS version, edition"
    ).single()
    print(f"Neo4j     : {info['version']} ({info['edition']} edition)")

    plugins = s.run(
        "SHOW PROCEDURES YIELD name "
        "WHERE name STARTS WITH 'n10s' OR name STARTS WITH 'gds' OR name STARTS WITH 'apoc' "
        "RETURN split(name,'.')[0] AS ns, count(*) AS n ORDER BY ns"
    ).data()
    print("플러그인  :", {p["ns"]: p["n"] for p in plugins})

    if not s.run("SHOW PROCEDURES YIELD name WHERE name='n10s.graphconfig.init' RETURN name").data():
        raise RuntimeError("Neosemantics(n10s)가 없다. 위 docker run 의 NEO4J_PLUGINS 를 확인하라.")
    print("\n준비 완료 — n10s.graphconfig.init 사용 가능")

교재 루트 : Alessandro Negro - Knowledge Graphs and LLMs in Action
config.ini: {'uri': 'bolt://localhost:7687', 'user': 'neo4j', 'password': 'password', 'encrypted': '0'}
Neo4j     : 5.26.28 (community edition)
플러그인  : {'apoc': 190, 'gds': 423, 'n10s': 55}

준비 완료 — n10s.graphconfig.init 사용 가능


## 책 리스팅 번호와 저장소 파일 이름의 대응

교재 저장소는 MEAP 시절 번호를 그대로 쓴다. 3장의 경우 **책 번호 = 파일 번호 − 3** 이다. 예를 들어 책의 *Listing 3.15 Configuring the Neosemantics plugin* 은 저장소에서 [`listings/3.18 - initialize_neo_semantics`](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/blob/main/chapters/ch03/listings/3.18%20-%20initialize_neo_semantics) 다. 이 오프셋은 챕터마다 다르므로(4장은 0) 다른 장에 그대로 적용하면 안 된다.

**업스트림 저장소의 버그 2건** 을 확인했고, 두 리스팅은 해설판 본문에서 가져와 보완했다.

| 책 리스팅 | 저장소 파일 | 문제 | 이 노트북의 처리 |
| --- | --- | --- | --- |
| 3.21 연관 찾기 | [`3.24 - explores_disease_phenotype_associations`](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/blob/main/chapters/ch03/listings/3.24%20-%20explores_disease_phenotype_associations) | `3.25` 파일과 내용이 **완전 중복** (엉뚱한 질의) | 해설판 본문 사용 |
| 3.25 제1형 당뇨병 표현형 | [`3.28 - phenotypes_associated_with_type_1_diabetes`](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/blob/main/chapters/ch03/listings/3.28%20-%20phenotypes_associated_with_type_1_diabetes) | **0바이트 빈 파일** | 해설판 본문 사용 |

또 해설판의 책 3.21 본문은 `MERGE (dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)` 로 시작하는데, 바인딩되지 않은 변수에 `MERGE` 를 쓰면 **노드와 관계를 새로 만들어 그래프를 오염시킨다.** 의도는 조회이므로 `MATCH` 로 바로잡아 실행한다.

In [2]:
"""리스팅 접근 헬퍼 — 책 번호로 저장소 리스팅을 찾아 보고, 실행한다."""
import time
from kgbook import cypher

LISTING_OFFSET = 3      # 책 3.15 == 파일 '3.18 - ...'  (3장 전용 오프셋)

# 업스트림 파일이 비어 있거나 중복이라 해설판 본문에서 가져온 리스팅.
# 3.21 은 해설판의 MERGE 를 MATCH 로 바로잡았다 (MERGE 는 그래프를 오염시킨다).
BOOK_CYPHER = {
    "3.21": """MATCH (dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label AS disease, collect(phe.label) AS features
LIMIT 3""",
    "3.25": """MATCH path=(dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
WHERE dis.id = "OMIM:222100"
RETURN path""",
}


def repo_number(book_no: str) -> str:
    """책 리스팅 번호 -> 저장소 파일 번호 접두사."""
    major, minor = book_no.split(".")
    return f"{major}.{int(minor) + LISTING_OFFSET}"


def source_of(book_no: str) -> str:
    """책 리스팅의 Cypher 원문. 저장소 파일이 망가진 것은 해설판 본문을 쓴다."""
    if book_no in BOOK_CYPHER:
        return BOOK_CYPHER[book_no]
    return cypher.read(repo_number(book_no), HERE / "listings").strip()


def show(book_no: str) -> None:
    origin = "해설판 본문" if book_no in BOOK_CYPHER else f"파일 {repo_number(book_no)}"
    print(f"── 책 Listing {book_no}  ({origin}) ──")
    print(source_of(book_no))


def run(book_no: str, limit: int = 5, db: str = DB):
    """책 리스팅을 실행하고 결과 일부를 표로 보여준다."""
    statements = [s.strip() for s in source_of(book_no).split(";") if s.strip()]
    started = time.time()
    rows = []
    with cypher.driver() as drv, drv.session(database=db) as s:
        for st in statements:
            rows = [r.data() for r in s.run(st)]
    print(f"책 {book_no}: {time.time() - started:.1f}초, {len(rows)}행")
    for r in rows[:limit]:
        print("   ", {k: str(v)[:60] for k, v in r.items()})
    return rows


def stats() -> dict:
    """현재 그래프 규모."""
    queries = {
        "Resource": "MATCH (n:Resource) RETURN count(n)",
        "HpoPhenotype": "MATCH (n:HpoPhenotype) RETURN count(n)",
        "HpoDisease": "MATCH (n:HpoDisease) RETURN count(n)",
        "HAS_PHENOTYPIC_FEATURE": "MATCH ()-[r:HAS_PHENOTYPIC_FEATURE]->() RETURN count(r)",
    }
    with cypher.driver() as drv, drv.session(database=DB) as s:
        return {k: s.run(q).single()[0] for k, q in queries.items()}


# 3장 리스팅 전체 대조표
print(f"{'책':<7}{'저장소 파일':<52}")
for path in cypher.listings(HERE / "listings"):
    file_no = path.name.split(" ")[0]
    major, minor = file_no.split(".")
    book_no = f"{major}.{int(minor) - LISTING_OFFSET}"
    note = "  <- 파일 손상, 해설판 사용" if book_no in BOOK_CYPHER else ""
    print(f"{book_no:<7}{path.name[:50]:<52}{note}")

print("\n현재 그래프:", stats())

책      저장소 파일                                              
3.13   3.16 - create_database                              
3.14   3.17 - create_db_constraints                        
3.15   3.18 - initialize_neo_semantics                     
3.16   3.19 - load_hpo_ontology                            
3.17   3.20 - enrich_resource_node_with_abnormalities      
3.18   3.21 - show_kg_at_the_current_stage                 
3.19   3.22 - create_disease_nodes                         
3.20   3.23 - create_rels_phenotypes_diseases              
3.21   3.24 - explores_disease_phenotype_associations        <- 파일 손상, 해설판 사용
3.22   3.25 - add_base_properties_to_rels                  
3.23   3.26 - enrich_with_descriptive_properties           
3.24   3.27 - remove_unused_nodes                          
3.25   3.28 - phenotypes_associated_with_type_1_diabetes     <- 파일 손상, 해설판 사용
3.26   3.29 - diseases_associated_with_specific_features   
3.27   3.30 - subclasses_of_abnormality_of_endocrine_syst  
3.28

---
## 3.1 준비 운동 — 데이터 이해

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/df51a084f481f21bb5ee1947d6c6dd046979f344bca4660083648e7def240b87.jpg" width="820" alt="데이터 이해 단계">

*그림 3.5 — 임상의 활동을 지원하기 위해 데이터를 이해하는 단계. 이 탐색적(explorative) 단계에서 지식 그래프를 구성하는 데 필요한 핵심 정보를 얻는다.*

데이터 소스는 **인간 표현형 온톨로지(Human Phenotype Ontology, HPO)** 저장소이고, 성격이 전혀 다른 두 갈래를 준다. 이 장의 기술적 난점 대부분이 "이 둘을 어떻게 하나로 합치나"에서 나온다.

| | ① 온톨로지 | ② 주석 |
| --- | --- | --- |
| 파일 | `hp.owl` | `phenotype.hpoa` |
| 형식 | RDF/XML (OWL) | 탭 구분 값 (TSV) |
| 담는 것 | 표현형 이상의 **표준 정의와 계층** | 질병 ↔ 표현형 특징의 **연관과 그 근거** |
| 적재 방법 | Neosemantics `n10s.rdf.import.fetch` | Cypher `LOAD CSV` |
| 책 리스팅 | 3.16 | 3.19\~3.23 |
| 원본 | [purl.obolibrary.org/obo/hp.owl](http://purl.obolibrary.org/obo/hp.owl) | [phenotype.hpoa (HPO releases)](https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/phenotype.hpoa) |

### ① `hp.owl` — 표준 정의와 계층

각 표현형 이상을 `owl:Class` 로 정의한다. 제1형 당뇨병을 예로 들면:

- URI `obo:HP_0100651` 로 식별된다

- 자연어 정의를 `obo:IAO_0000115` 에 담는다

- 작성자·작성일 메타데이터를 갖는다 (`created_by`, `creation_date`)

- 외부 DB 상호참조 `hasDbXref` — `MSH:D003922`, `SNOMEDCT_US:46635009`, `UMLS:C0011854`

- 동의어 `hasExactSynonym` — "Diabetes mellitus Type I", "Juvenile diabetes mellitus"

- **`rdfs:subClassOf obo:HP_0000819`** — 상위 개념이 당뇨병임을 선언한다

마지막 항목이 결정적이다. 이 **`subClassOf` 계층이 3.5절 추론의 재료** 가 된다.

### ② `phenotype.hpoa` — 연관과 근거

각 행은 "이 질병에 이 표현형 특징이 있다"는 한 건의 주석이고, 그 **근거** 를 함께 담는다.

| 필드 | 예시 | 뜻 |
| --- | --- | --- |
| `database_id` | `OMIM:222100` | 질병 식별자 (OMIM·Orphanet 등) |
| `disease_name` | `Diabetes mellitus, insulin-dependent-1` | 질병 이름 |
| `hpo_id` | `HP:0410050` | 연관된 표현형 이상의 HPO 식별자 |
| `reference` | `PMID:9357814;...` | 근거 출처 (PubMed ID) |
| `evidence` | `PCS` | 증거 수준 (PCS=발표된 임상 연구, IEA=전자 주석 추론, TAS=추적 가능한 저자 진술) |
| `frequency` | `30/30` | 해당 질병 환자 30명 중 30명에서 관찰 |
| `aspect` | `P` | 표현형 이상(phenotypic abnormality) |
| `biocuration` | `HPO:NicoleVasilevsky[2018-02-23]` | 주석 작성자와 날짜 |

`reference`·`evidence`·`biocuration` 이 있다는 점이 결정적이다. 이것들은 **질병이나 표현형에 붙는 속성이 아니라 둘을 잇는 관계에 붙어야 하는 속성** 이다. 3.2절의 기술 선택이 여기서 갈린다.

In [3]:
"""실습 3-A — 책 Listing 3.4 재현: phenotype.hpoa 원본을 직접 들여다본다.

책은 이 파일의 표본만 보여준다. 실제 파일을 내려받아 메타데이터·헤더·데이터 행을
구분해 확인하고, 위 표의 필드 설명과 대조해 본다.
"""
import urllib.request

HPOA_URL = ("https://github.com/obophenotype/human-phenotype-ontology"
            "/releases/latest/download/phenotype.hpoa")

# 파일 전체는 수십 MB 다. 앞부분만 읽는다.
with urllib.request.urlopen(HPOA_URL) as response:
    head = response.read(4000).decode("utf-8", errors="replace")

lines = head.splitlines()
meta = [l for l in lines if l.startswith("#")]
header_index = len(meta)
columns = lines[header_index].split("\t")
first_row = lines[header_index + 1].split("\t")

print(f"=== 메타데이터 {len(meta)}줄 ===")
for line in meta:
    print("   ", line)

print(f"\n=== 헤더 (줄 {header_index + 1}) — 컬럼 {len(columns)}개 ===")
print("   ", " | ".join(columns))

print("\n=== 첫 데이터 행 — 빈 필드는 생략 ===")
for i, (name, value) in enumerate(zip(columns, first_row)):
    if value:
        print(f"    row[{i}]  {name:<13} = {value[:70]}")

print(f"\n※ 메타데이터 {len(meta)}줄 + 헤더 1줄 = {header_index + 1}줄.")
print("  책 3.19~3.23 의 LOAD CSV 는 헤더를 쓰지 않으므로 SKIP 5 로 이 5줄을 건너뛴다.")
print("\n※ 책은 2025년 2월 HPO 로 결과를 냈다. 위 #version 이 그보다 최신이면")
print("  숫자와 상위 질병 목록이 책과 조금씩 다를 수 있다 (데이터 드리프트).")

=== 메타데이터 4줄 ===
    #description: "HPO annotations for rare diseases [8574: OMIM; 47: DECIPHER; 4337 ORPHANET]"
    #version: 2026-06-23
    #tracker: https://github.com/obophenotype/human-phenotype-ontology/issues
    #hpo-version: http://purl.obolibrary.org/obo/hp/releases/2026-06-23/hp.json

=== 헤더 (줄 5) — 컬럼 12개 ===
    database_id | disease_name | qualifier | hpo_id | reference | evidence | onset | frequency | sex | modifier | aspect | biocuration

=== 첫 데이터 행 — 빈 필드는 생략 ===
    row[0]  database_id   = OMIM:619340
    row[1]  disease_name  = Developmental and epileptic encephalopathy 96
    row[3]  hpo_id        = HP:0011097
    row[4]  reference     = PMID:31675180
    row[5]  evidence      = PCS
    row[7]  frequency     = 1/2
    row[10]  aspect        = P
    row[11]  biocuration   = HPO:probinson[2021-06-21]

※ 메타데이터 4줄 + 헤더 1줄 = 5줄.
  책 3.19~3.23 의 LOAD CSV 는 헤더를 쓰지 않으므로 SKIP 5 로 이 5줄을 건너뛴다.

※ 책은 2025년 2월 HPO 로 결과를 냈다. 위 #version 이 그보다 최신이면
  숫자와 상위 질병 목록이 책과 조금씩 다를 수 있다 (

---
## 3.2 RDF vs LPG — 왜 LPG를 골랐나

KG를 만드는 두 대표 기술이다.

**RDF (Resource Description Framework)** 는 W3C 표준이다. 모든 문장이 **트리플** — 주어(노드)·술어(관계)·목적어(노드) — 이고, 지식 그래프를 문장들의 모음으로 모델링한다. **온톨로지를 만드는 데 특히 적합** 하다. HPO가 `.owl`(Web Ontology Language, RDF 직렬화)로 배포되는 이유가 이것이다. OWL 온톨로지는 널리 쓰여서 GPT·Claude 같은 LLM도 이것으로 학습되었고, 그래서 LLM이 OWL 기반 데이터를 해석·추론하기가 더 쉽다.

**LPG (Labeled Property Graph)** 는 노드와 관계에 **키–값 쌍** 을 붙인다. 빠른 질의 기반 순회와 경로 분석에 강하다.

결정적 차이는 **간선에 정보를 붙이는 능력** 이다.

- RDF에서 관계는 **전역적으로** 정의된다. 어떤 술어에 붙인 메타데이터는 그래프 전체의 그 관계 **모든 인스턴스** 에 영향을 준다

- LPG는 노드 사이에 **고유한 간선** 을 허용한다. 개별 관계마다 별도의 속성을 붙일 수 있다

### 표 한 행이 그래프 간선 하나가 된다

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/0a8768ea67bd7cd8ce084f51410c21f4e0f6d5f017a74a15e2432cf78197c284.jpg" width="820" alt="표 한 행에서 KG 간선으로의 변환">

*그림 3.6 — 표의 한 행에서 KG 간선으로의 데이터 변환. 질병(`OMIM:222100`)과 표현형 특징(`HP:0410050`)은 **노드** 가 되고, 주석 작성자·생성일·출처 정보는 `HAS_PHENOTYPIC_FEATURE` **간선의 속성** 이 된다. 이 그림이 3.2절 결론을 한 장으로 요약한다.*

3.1절에서 본 주석 데이터는 "이 논문에서, 이 증거 수준으로, 이 사람이 이 날짜에" 라는 정보가 **주석 한 건마다 다르다.** 같은 질병–표현형 쌍에 대해서도 근거가 여러 건 있을 수 있다. 그래서 이 장은 **LPG를 고른다.**

### 3.2.2 RDF에서 간선 속성을 표현하는 세 우회로

| 방식 | 어떻게 | 대가 |
| --- | --- | --- |
| **N항 관계** (책 3.5\~3.6) | 주석을 나타내는 **새 개념(노드)** 을 만들어 질병과 표현형 사이에 끼운다. 흔히 **공백 노드(blank node)** — 전역 식별자 없는 이름 없는 자원 — 를 쓴다 | 모델이 한 단계 깊어지고 질의가 길어진다 |
| **명명된 그래프** (책 3.7\~3.8) | 트리플 묶음을 하나의 개체로 다뤄 맥락별 정보를 붙인다. 트리플에 네 번째 요소를 더하는 셈 | 표준 트리플 모델을 벗어난다 |
| **RDF-star** (책 3.9\~3.10) | 트리플 **자체** 에 속성을 붙이는 RDF 확장. RDF-DEV 커뮤니티 그룹이 RDF와 LPG를 화해시키려 작업 중 | 아직 확장 명세이고 지원이 고르지 않다 |

**LPG** (책 3.11\~3.12)는 관계 안에 속성을 곧바로 담으므로 우회가 필요 없다. 대신 LPG는 RDF의 고급 의미론을 표현하지 못하는데, 이 간극을 [**Neosemantics(n10s)**](https://neo4j.com/labs/neosemantics/) 플러그인이 메운다 — Neo4j 안에서 OWL·RDFS·SKOS 어휘로 기본적인 추론을 실행하게 해 준다. 3.5절이 바로 그 조합이다.

In [4]:
"""실습 3-B — 해설판 3.2.1의 연습문제: 요구사항으로 기술을 골라 본다.

해설판은 "정답은 하나가 아니다" 라고 하지만, 요구사항을 하나씩 두 기술에
대조해 보면 왜 이 장이 LPG 를 고르는지가 드러난다.
"""
REQUIREMENTS = [
    ("임상의는 데이터가 어떻게 모델링되는지에 관심이 없다",
     "무관", "무관", "기술 선택의 근거가 되지 못함"),
    ("표현형 특징의 모호하지 않은 표현, 가능하면 계층 구조",
     "강함", "가능", "RDF/OWL 이 계층 정의에 강하다. LPG 는 n10s 로 보완"),
    ("주석 한 건마다 출처·작성자·날짜를 기록",
     "우회 필요", "직접 지원", "★ 여기서 갈린다 — LPG 는 간선에 속성을 바로 붙인다"),
    ("같은 표현형-질병 쌍에 근거가 여러 건일 수 있다",
     "우회 필요", "직접 지원", "★ LPG 는 노드 쌍 사이에 고유한 간선 여러 개 허용"),
    ("특정 표현형과 연관된 모든 사례를 쉽게 비교",
     "가능", "강함", "LPG 는 질의 기반 순회·경로 분석에 강하다"),
]

print(f"{'요구사항':<46}{'RDF':<12}{'LPG':<12}비고")
print("-" * 118)
for req, rdf, lpg, note in REQUIREMENTS:
    print(f"{req[:44]:<46}{rdf:<12}{lpg:<12}{note}")

print("\n결론: '주석마다 다른 메타데이터' 와 '같은 쌍에 여러 근거' 두 요구가")
print("      간선 속성을 필수로 만들고, 그래서 이 장은 LPG(Neo4j)를 고른다.")
print("      RDF/OWL 의 계층 정의 능력은 n10s 로 LPG 안에 끌어온다 (3.3, 3.5절).")

요구사항                                          RDF         LPG         비고
----------------------------------------------------------------------------------------------------------------------
임상의는 데이터가 어떻게 모델링되는지에 관심이 없다                  무관          무관          기술 선택의 근거가 되지 못함
표현형 특징의 모호하지 않은 표현, 가능하면 계층 구조                강함          가능          RDF/OWL 이 계층 정의에 강하다. LPG 는 n10s 로 보완
주석 한 건마다 출처·작성자·날짜를 기록                        우회 필요       직접 지원       ★ 여기서 갈린다 — LPG 는 간선에 속성을 바로 붙인다
같은 표현형-질병 쌍에 근거가 여러 건일 수 있다                   우회 필요       직접 지원       ★ LPG 는 노드 쌍 사이에 고유한 간선 여러 개 허용
특정 표현형과 연관된 모든 사례를 쉽게 비교                      가능          강함          LPG 는 질의 기반 순회·경로 분석에 강하다

결론: '주석마다 다른 메타데이터' 와 '같은 쌍에 여러 근거' 두 요구가
      간선 속성을 필수로 만들고, 그래서 이 장은 LPG(Neo4j)를 고른다.
      RDF/OWL 의 계층 정의 능력은 n10s 로 LPG 안에 끌어온다 (3.3, 3.5절).


---
## 3.3 지식 그래프 짓기

과정은 두 단계다. **① 온톨로지를 적재** 하고, 그 온톨로지를 기준 삼아 **② 주석 데이터를 수집** 한다.

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/a522478921eea8892f0345b5359b3df899369b7d634f78931d9daf1bdf037d14.jpg" width="820" alt="온톨로지 수집 및 처리 단계">

*그림 3.7 — 온톨로지 수집 및 처리(Ontology ingestion and processing). 이 노트북의 책 3.13\~3.18 셀들이 이 흐름을 그대로 따른다.*

| 책 리스팅 | 하는 일 | 실측 소요 |
| --- | --- | --- |
| 3.13 | `CREATE DATABASE hpo` | *Community 에디션에서 불가 — 건너뜀* |
| 3.14 | 제약조건·인덱스 생성 | 0.2초 |
| 3.15 | Neosemantics 초기 설정 | 0.2초 |
| 3.16 | `hp.owl` 어휘 적재 | **9.5초 / 926,450 트리플** |
| 3.17 | `HpoPhenotype` 레이블·`id` 부여 | 0.5초 |
| 3.18 | 현재 KG 일부 확인 | 0.2초 |
| 3.19 | `HpoDisease` 노드 생성 | 2.8초 |
| 3.20 | 질병 ↔ 표현형 관계 생성 | 3.3초 |
| 3.21 | 연관 확인 (표 3.1) | 0.1초 |
| 3.22 | 관계에 기본 속성 추가 | **약 8분 (496초)** |
| 3.23 | 관계에 서술 속성 추가 | 7.2초 |
| 3.24 | 불필요 노드 정리 | 5초 / 207,754개 삭제 |

> **책 3.22가 8분 걸린다.** 285,000개 관계에 대해 `FOREACH` 8개를 배치 없이 돌리기 때문이다. 업스트림 코드를 그대로 두었으니 시간을 감안하라.

> **순서가 중요하다.** 해설판의 경고대로, 책 3.18의 질의는 **3.24 정리 단계 이전에만** 작동한다. 정리가 온톨로지 메타데이터 노드를 지우기 때문이다. 저장소 코드로 전체를 한 번에 돌리면 3.18은 실패한다.

In [5]:
"""구축 단계 실행 헬퍼.

그래프가 이미 구축돼 있으면 건너뛴다. 처음부터 다시 만들려면 FORCE_REBUILD=True
로 바꾸고 아래 셀들을 순서대로 실행한다 (총 9분 남짓).
"""
FORCE_REBUILD = False


def build(*book_numbers: str, guard: str = "HpoDisease", threshold: int = 0) -> None:
    """구축용 리스팅을 실행한다. guard 카운트가 이미 차 있으면 건너뛴다."""
    current = stats()[guard]
    if current > threshold and not FORCE_REBUILD:
        print(f"이미 구축됨 ({guard}={current:,}) — 건너뜀. "
              f"다시 만들려면 FORCE_REBUILD=True")
        return
    for number in book_numbers:
        run(number, limit=1)


print("현재 그래프:", {k: f"{v:,}" for k, v in stats().items()})
print(f"FORCE_REBUILD = {FORCE_REBUILD}")

현재 그래프: {'Resource': '33,369', 'HpoPhenotype': '20,413', 'HpoDisease': '12,956', 'HAS_PHENOTYPIC_FEATURE': '284,994'}
FORCE_REBUILD = False


In [6]:
"""책 3.13 + 3.14 — 데이터베이스와 스키마 준비.

3.13 은 Community 에디션에서 실행할 수 없으므로 원문만 확인하고 넘어간다.
3.14 의 제약조건은 단순한 무결성 장치가 아니다. Resource.uri 유일성 제약이
있어야 n10s 가 온톨로지를 MERGE 로 멱등하게 적재할 수 있다.
"""
show("3.13")
print("\n→ Community 에디션은 멀티 데이터베이스를 지원하지 않아 실행할 수 없다.")
print(f"  이 노트북은 기본 '{DB}' 데이터베이스를 쓴다.\n")

show("3.14")
run("3.14", limit=1)

with cypher.driver() as drv, drv.session(database=DB) as s:
    print("\n적용된 제약조건/인덱스:")
    for r in s.run("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties").data():
        print(f"    제약 {r['name']:<24} {r['labelsOrTypes']} {r['properties']}")
    for r in s.run("SHOW INDEXES YIELD name, labelsOrTypes, properties "
                   "WHERE labelsOrTypes IS NOT NULL AND "
                   "any(l IN labelsOrTypes WHERE l STARTS WITH 'Hpo')").data():
        print(f"    인덱스 {r['name']:<22} {r['labelsOrTypes']} {r['properties']}")

── 책 Listing 3.13  (파일 3.16) ──
CREATE DATABASE hpo IF NOT EXISTS

→ Community 에디션은 멀티 데이터베이스를 지원하지 않아 실행할 수 없다.
  이 노트북은 기본 'neo4j' 데이터베이스를 쓴다.

── 책 Listing 3.14  (파일 3.17) ──
CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS FOR (r:Resource) REQUIRE r.uri IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Resource) REQUIRE (n.id) IS UNIQUE;
CREATE INDEX disease_id IF NOT EXISTS FOR (n:HpoDisease) ON (n.id);
CREATE INDEX phenotype_id IF NOT EXISTS FOR (n:HpoPhenotype) ON (n.id);
책 3.14: 0.0초, 0행

적용된 제약조건/인덱스:
    제약 n10s_unique_uri          ['Resource'] ['uri']
    제약 resource_id              ['Resource'] ['id']
    인덱스 disease_id             ['HpoDisease'] ['id']
    인덱스 phenotype_id           ['HpoPhenotype'] ['id']


In [7]:
"""책 3.15 + 3.16 — Neosemantics 설정과 HPO 어휘 적재.

3.15 의 두 설정이 뒤에 나오는 모든 Cypher 의 모양을 결정한다.
  handleVocabUris:"IGNORE"  -> 임포트 시 네임스페이스를 버린다. 그래서 노드
                               속성이 'obo:IAO_0000115' 가 아니라 'iAO_0000115'
  applyNeo4jNaming:True     -> 관계 타입을 대문자로. 그래서 rdfs:subClassOf 가
                               LPG 관용 표기인 :SUBCLASSOF 로 들어온다
3.16 은 hp.owl 을 그대로 가져와 트리플을 적재한다. 책은 899,558 개였다.
"""
show("3.15")
build("3.15", "3.16", guard="Resource")

with cypher.driver() as drv, drv.session(database=DB) as s:
    print("\n적재된 관계 타입 상위 5개 (대문자 변환 확인):")
    for r in s.run("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n "
                   "ORDER BY n DESC LIMIT 5").data():
        print(f"    {r['type']:<24} {r['n']:>9,}")

── 책 Listing 3.15  (파일 3.18) ──
CALL n10s.graphconfig.init();
CALL n10s.graphconfig.set({ handleVocabUris: "IGNORE" });
CALL n10s.graphconfig.set({ applyNeo4jNaming: True });
이미 구축됨 (Resource=33,369) — 건너뜀. 다시 만들려면 FORCE_REBUILD=True

적재된 관계 타입 상위 5개 (대문자 변환 확인):
    HAS_PHENOTYPIC_FEATURE     284,994
    SUBCLASSOF                  24,378
    IAO_0100001                      1


### 적재된 온톨로지는 어떻게 생겼나

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/e62afc30c4a44268378b02b1f3daf3fdf1b53b17267f22c8a4af1468c73a17d7.jpg" width="820" alt="적재된 HPO 온톨로지의 일부">

*그림 3.8 — LPG를 저장 모델로 삼아 그래프 데이터베이스에 적재된 HPO 온톨로지의 일부. 두 종류의 정보가 구별된다. **왼쪽은 온톨로지 정보** (노드의 성격·동의어 타입·주석 출처 등 메타데이터), **오른쪽은 표현형 특징 관련 도메인 정보** (당뇨병 계층 구조)다. 다음 셀의 책 3.18 질의가 이 화면을 만든다.*

책 3.18은 이 그림을 재현하는 질의다. 세 경로를 함께 조회한다.

- `path1` — "Diabetes mellitus"의 **하위 클래스** 경로

- `path2` — 관련 **주석 소스** 경로 (`ANNOTATEDSOURCE`)

- `path3` — **동의어 타입** 등의 경로 (`ANNOTATEDPROPERTY`, `HASSYNONYMTYPE`)

그림 3.8의 왼쪽 절반이 `path2`·`path3` 이고, 이 노드들이 **3.24 정리 단계에서 삭제된다.** 그래서 정리 후에는 이 질의가 빈 결과를 낸다.

In [8]:
"""책 3.17 + 3.18 — 노드 보강과 중간 점검.

3.17 은 uri 가 HPO 네임스페이스로 시작하는 Resource 에 HpoPhenotype 레이블을
붙이고, uri 를 가공해 'HP:0000001' 형태의 id 를 만든다. coalesce 로 기존 id 는
보존한다. 이 id 가 다음 절에서 TSV 의 hpo_id 와 맞물리는 접합면이다.
"""
show("3.17")
build("3.17", guard="HpoPhenotype")

print()
show("3.18")
rows = run("3.18", limit=1)
if not rows:
    print("\n※ 빈 결과다 — 그림 3.8 의 왼쪽 절반(온톨로지 메타데이터)이 이미 삭제된 상태다.")
    print("  책 3.24 정리 단계가 실행된 그래프다. 이 질의 결과를 보려면")
    print("  FORCE_REBUILD=True 로 처음부터 다시 만들어야 한다.")

── 책 Listing 3.17  (파일 3.20) ──
MATCH (n:Resource) 
WHERE n.uri STARTS WITH "http://purl.obolibrary.org/obo/HP" 
SET n:HpoPhenotype, 
       n.id = coalesce(n.id,
   replace(apoc.text.replace(n.uri,'(.*)obo/',''),'_', ':'))


이미 구축됨 (HpoPhenotype=20,413) — 건너뜀. 다시 만들려면 FORCE_REBUILD=True

── 책 Listing 3.18  (파일 3.21) ──
MATCH path1=(n:HpoPhenotype)<-[:SUBCLASSOF]-(m:HpoPhenotype)
WHERE n.label = "Diabetes mellitus"
WITH path1
MATCH path2=(i:HpoPhenotype)<-[:ANNOTATEDSOURCE]-(j)
WHERE i.label in ["Diabetes mellitus", "Type I diabetes mellitus"]
WITH path1, path2, j
MATCH path3=(j)-[:ANNOTATEDPROPERTY|HASSYNONYMTYPE]-()
RETURN path1, path2, path3


책 3.18: 0.0초, 0행

※ 빈 결과다 — 그림 3.8 의 왼쪽 절반(온톨로지 메타데이터)이 이미 삭제된 상태다.
  책 3.24 정리 단계가 실행된 그래프다. 이 질의 결과를 보려면
  FORCE_REBUILD=True 로 처음부터 다시 만들어야 한다.


### 3.3.2 주석 수집 — 두 소스가 합쳐지는 지점

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/747a0d2a35fa40a70a18f4868d0c975b5771eeac7f731d1d636bbb159dcd0c59.jpg" width="820" alt="주석 데이터셋 수집 단계">

*그림 3.9 — 지식 그래프 구축을 마무리하기 위해 주석 데이터셋을 수집·처리하는 두 번째 단계. 그림 3.7의 온톨로지 적재에 이어지는 후반부다.*

지금까지는 온톨로지만 들어왔다. 이제 TSV 주석 파일을 넣어 그래프를 완성한다. 이 단계가 이 장의 핵심이다. **RDF/XML 로 만든 온톨로지 노드와 TSV 로 만든 질병 노드가 `id` 값을 매개로 이어진다.**

- 3.17 이 온톨로지 노드에 `HP:0410050` 형태의 `id` 를 만들어 두었다

- 3.19 가 TSV 의 `row[0]`(`OMIM:222100`)으로 `HpoDisease` 노드를 만든다

- 3.20 이 `row[0]` 과 `row[3]`(`HP:0410050`)로 두 노드를 찾아 `:HAS_PHENOTYPIC_FEATURE` 관계를 만든다

- 3.22\~3.23 이 그 관계에 `row[4]`\~`row[11]` 을 속성으로 붙인다 — 그림 3.6이 보여준 변환이 실제로 일어나는 지점이다

즉 **형식이 다른 두 파일이 온톨로지의 표준 식별자를 공통 언어로 삼아 하나의 그래프가 된다.** 이것이 3.1절에서 말한 의미 통합의 구체적 모습이다.

HPOA 파일이 담는 가치 있는 정보를 다시 정리하면:

- 질병과 여러 표현형 특징 사이의 **명시적 연관**

- 그 연관을 뒷받침하는 **증거** — 전자 주석 추론(IEA), 발표된 임상 연구(PCS), 추적 가능한 저자 진술(TAS)

- **발병 연령** 과 함께 나타나는 **빈도**

- 온톨로지 출처를 서술하는 **추가 메타데이터**

In [9]:
"""책 3.19 + 3.20 — 질병 노드와 관계 생성.

3.19: MERGE 로 질병 노드를 만들고, 처음 만들 때만 label 을 채운다(ON CREATE).
      같은 질병이 파일에 여러 행 등장하므로 MERGE 가 필수다.
3.20: 같은 파일을 다시 읽어 두 id 로 노드를 찾아 관계를 만든다.
      Resource:HpoDisease 이중 레이블이라 3.14 의 id 유일성 제약이 걸린다.
"""
show("3.19")
print()
show("3.20")
build("3.19", "3.20", guard="HpoDisease")

print("\n그래프 규모:", {k: f"{v:,}" for k, v in stats().items()})

── 책 Listing 3.19  (파일 3.22) ──
LOAD CSV FROM 'https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/phenotype.hpoa' AS row
FIELDTERMINATOR '\t'
WITH row
SKIP 5  // #A
MERGE (dis:Resource:HpoDisease {id: row[0]})
ON CREATE SET dis.label = row[1]

── 책 Listing 3.20  (파일 3.23) ──
LOAD CSV FROM 'https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/phenotype.hpoa' AS row
FIELDTERMINATOR '\t'
WITH row
SKIP 5
MATCH (dis:HpoDisease)
WHERE dis.id = row[0]
MATCH (phe:HpoPhenotype)
WHERE phe.id = row[3]
MERGE (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe)
이미 구축됨 (HpoDisease=12,956) — 건너뜀. 다시 만들려면 FORCE_REBUILD=True

그래프 규모: {'Resource': '33,369', 'HpoPhenotype': '20,413', 'HpoDisease': '12,956', 'HAS_PHENOTYPIC_FEATURE': '284,994'}


In [10]:
"""책 3.21 — 통합 결과 확인 (책 표 3.1 재현).

저장소 파일 '3.24 - explores_disease_phenotype_associations' 는 '3.25' 파일과
내용이 완전히 중복된 업스트림 버그다. 그래서 해설판 본문을 쓴다.
해설판 본문은 MERGE 로 시작하는데, 바인딩되지 않은 변수에 MERGE 를 쓰면 노드를
새로 만들어 버리므로 MATCH 로 바로잡았다 (BOOK_CYPHER 참고).
"""
show("3.21")

before = stats()
rows = run("3.21", limit=0)
after = stats()

print("\n질병별 연관 표현형 (책 표 3.1 과 대조):")
for r in rows:
    print(f"    {r['disease'][:56]:<58} 특징 {len(r['features']):>2}개")

print(f"\n노드 수 변화: {before['HpoDisease']:,} -> {after['HpoDisease']:,} "
      f"(MATCH 이므로 0 이어야 한다)")

── 책 Listing 3.21  (해설판 본문) ──
MATCH (dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label AS disease, collect(phe.label) AS features
LIMIT 3


책 3.21: 0.1초, 3행

질병별 연관 표현형 (책 표 3.1 과 대조):
    Developmental and epileptic encephalopathy 96              특징 11개
    Pseudohyperkalemia, familial, 2, due to red cell leak      특징  7개
    Immunoglobulin kappa light chain deficiency                특징  7개

노드 수 변화: 12,956 -> 12,956 (MATCH 이므로 0 이어야 한다)


In [11]:
"""책 3.22 + 3.23 — 관계에 속성 붙이기. 3.2절에서 LPG를 고른 이유가 실현되는 지점.

3.22: row[4]~row[11] 을 source·evidence·onset·frequency·sex·modifier·aspect·
      biocuration 속성으로 관계에 붙인다. FOREACH(_ IN CASE WHEN ... THEN [1]
      ELSE [] END | SET ...) 는 "값이 있을 때만 SET" 을 표현하는 Cypher 관용구다
      (Cypher 에는 조건부 SET 문법이 없다).
3.23: apoc.periodic.iterate 로 배치 처리하며, 증거 코드에 사람이 읽을 이름·설명과
      PubMed/OMIM 열람 URL 을 더 붙인다 (evidenceName, evidenceDescription, url 등).

★ 3.22 는 285,000 개 관계에 FOREACH 8개를 배치 없이 돌려 약 8분 걸린다.
"""
with cypher.driver() as drv, drv.session(database=DB) as s:
    tagged = s.run("MATCH ()-[r:HAS_PHENOTYPIC_FEATURE]->() "
                   "WHERE r.evidence IS NOT NULL RETURN count(r) AS n").single()["n"]

if tagged > 0 and not FORCE_REBUILD:
    print(f"이미 속성이 붙어 있다 (evidence 보유 관계 {tagged:,}개) — 건너뜀.")
else:
    print("약 8분 걸린다...")
    run("3.22", limit=1)
    run("3.23", limit=1)

with cypher.driver() as drv, drv.session(database=DB) as s:
    print("\n관계에 실제로 붙은 속성 (같은 질병-표현형 쌍에 근거가 여러 건일 수 있다):")
    for r in s.run("""MATCH (d:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(p:HpoPhenotype)
                      WHERE r.evidence IS NOT NULL AND r.frequency IS NOT NULL
                      RETURN d.label AS disease, p.label AS phenotype,
                             r.source AS source, r.evidence AS evidence,
                             r.frequency AS frequency
                      LIMIT 3""").data():
        print(f"    {r['disease'][:34]:<36} -> {r['phenotype'][:26]:<28}")
        print(f"       출처={r['source']}  증거={r['evidence']}  빈도={r['frequency']}")

이미 속성이 붙어 있다 (evidence 보유 관계 284,994개) — 건너뜀.

관계에 실제로 붙은 속성 (같은 질병-표현형 쌍에 근거가 여러 건일 수 있다):
    Developmental and epileptic enceph   -> EEG with burst suppression  
       출처=PMID:31675180  증거=PCS  빈도=2/2
    Developmental and epileptic enceph   -> Small for gestational age   
       출처=PMID:31675180  증거=PCS  빈도=1/2
    Developmental and epileptic enceph   -> Primary microcephaly        
       출처=PMID:31675180  증거=PCS  빈도=1/2


In [12]:
"""책 3.24 — KG 정리. 온톨로지에서 왔지만 목적에 필요 없는 노드를 지운다.

apoc.periodic.iterate 로 배치 처리한다. HpoPhenotype 도 HpoDisease 도 아닌
Resource 노드를 DETACH DELETE 한다 (관계까지 함께 삭제).

★ 이 단계 이후 책 3.18 의 질의는 빈 결과를 낸다 — 해설판이 경고한 그 지점이다.
  그림 3.8 왼쪽의 온톨로지 메타데이터 노드가 사라지기 때문이다.
"""
show("3.24")

before = stats()
if before["Resource"] > before["HpoPhenotype"] + before["HpoDisease"] or FORCE_REBUILD:
    run("3.24", limit=1)
else:
    print("\n이미 정리됨 — 건너뜀.")

print("\n정리 전후:")
for key, value in before.items():
    print(f"    {key:<24} {value:>9,} -> {stats()[key]:>9,}")

── 책 Listing 3.24  (파일 3.27) ──
CALL apoc.periodic.iterate(
    "MATCH (n:Resource) RETURN id(n) as id",
    "MATCH (n)
     WHERE id(n) = id AND
           NOT 'HpoPhenotype' in labels(n) AND
           NOT 'HpoDisease' in labels(n)
     DETACH DELETE n",
     {batchSize:10000})
YIELD batches, total return batches, total

이미 정리됨 — 건너뜀.

정리 전후:
    Resource                    33,369 ->    33,369
    HpoPhenotype                20,413 ->    20,413
    HpoDisease                  12,956 ->    12,956
    HAS_PHENOTYPIC_FEATURE     284,994 ->   284,994


---
## 3.4 데이터 질의하기 — 임상의의 진단 시나리오

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/2ab1dbd408a0b1c0cc5fd7c9c0c01fd251a6613b79d9df13d969cf2e1b72bb7c.jpg" width="820" alt="생성된 KG에 질의하는 단계">

*그림 3.10 — 임상의 활동을 지원하기 위해 생성된 KG에 질의하기(Querying the generated KG). 그림 3.1 멘탈 모델의 마지막 단계다.*

해설판의 시나리오를 그대로 따라가 본다.

한 임상의가 **제1형 당뇨병을 앓는 소년** 을 진료한다. 이 환자의 임상 이력은 병원 데이터베이스에 **전자 건강 기록(EHR)** 으로 저장된다. 이 병원은 지식 그래프 패러다임을 받아들였으므로 환자 정보가 HPO·OMIM 용어로 저장된다. 그림 3.4에서 본 이중성 때문에 제1형 당뇨병은 두 ID로 기록된다.

- [`HP:0100651`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0100651) — **표현형 특징** 으로서

- [`OMIM:222100`](https://www.omim.org/entry/222100) — **질병** 으로서

임상의는 먼저 제1형 당뇨병의 전형적 표현형 특징들을 확인한다(**책 3.25**).

<img src="https://raw.githubusercontent.com/restful3/ds4th_study/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/images/51d02656dec871b6d826c3b53faaa19fc28025e37e5e8524336ec7a7e89cc5bb.jpg" width="760" alt="제1형 당뇨병 관련 표현형 특징 질의 결과">

*그림 3.11 — 제1형 당뇨병과 관련된 모든 표현형 특징을 가져오는 질의(책 3.25)의 결과. 중심 노드가 제1형 당뇨병이고 주변 노드들이 연관된 표현형 특징이다.*

그런데 진료 중 임상의는 제1형 당뇨병에 직접 연결되지 **않은** 새 증상 넷을 발견한다.

| 증상 | HPO ID |
| --- | --- |
| 성장 지연 (Growth delay) | [`HP:0001510`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0001510) |
| 큰 무릎 (Large knee) | [`HP:0030866`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0030866) |
| 감각신경성 청력 손상 (Sensorineural hearing impairment) | [`HP:0000407`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0000407) |
| 가려움증 (Pruritus) | [`HP:0000989`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0000989) |

> 해설판은 이 용어들을 `hpo.jax.org/app/browse/term/...` 로 링크하지만 HPO 사이트가 URL 체계를 바꿔 지금은 404 다. 위 링크는 같은 용어를 **EBI OLS4** 에서 열며, 이 장의 주제인 하위 클래스 계층도 함께 보여준다.

이제 질문이 뒤집힌다. "이 질병의 증상은?" 이 아니라 **"이 증상 조합을 가진 질병은?"** 이다. 그래프에서는 방향만 바꿔 순회하면 되는 질의다(**책 3.26**). 특징이 몇 개 맞아떨어지는지 세어 정렬하면 후보 질병이 순위로 나온다.


In [13]:
"""책 3.25 + 3.26 — 진단 질의.

3.25 는 저장소 파일이 0바이트 빈 파일이라 해설판 본문을 쓴다 (그림 3.11 재현).
3.26 은 다섯 표현형에 걸리는 질병을 모아 매칭 개수로 정렬한다 (책 표 3.2).
"""
show("3.25")
paths = run("3.25", limit=0)
print(f"→ OMIM:222100 (제1형 당뇨병) 에 연결된 표현형 특징 {len(paths)}개 — 그림 3.11 의 주변 노드들")

with cypher.driver() as drv, drv.session(database=DB) as s:
    for r in s.run("""MATCH (d:HpoDisease {id:"OMIM:222100"})-[:HAS_PHENOTYPIC_FEATURE]->(p)
                      RETURN p.label AS f ORDER BY f""").data():
        print(f"      - {r['f']}")

print("\n" + "=" * 78)
show("3.26")
rows = run("3.26", limit=0)

print("\n증상 조합에 맞는 질병 순위 (책 표 3.2 와 대조):")
for r in rows:
    print(f"    {r['num_of_features']}개  {r['disease_id']:<14} {r['disease_name'][:52]}")
print("\n→ 최상위 후보가 진단으로 이어진다.")

── 책 Listing 3.25  (해설판 본문) ──
MATCH path=(dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
WHERE dis.id = "OMIM:222100"
RETURN path
책 3.25: 0.0초, 8행
→ OMIM:222100 (제1형 당뇨병) 에 연결된 표현형 특징 8개 — 그림 3.11 의 주변 노드들
      - Autoimmunity
      - Decreased circulating 1,5 anhydroglucitol concentration
      - Diabetes mellitus
      - Hyperglycemia
      - Ketoacidosis
      - Polydipsia
      - Polyphagia
      - Polyuria

── 책 Listing 3.26  (파일 3.29) ──
MATCH (phe:HpoPhenotype)
WHERE phe.label in ["Growth delay", "Large knee", "Sensorineural hearing impairment", "Pruritus", "Type I diabetes mellitus"]
WITH phe
MATCH path=(dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe)
UNWIND dis as nodes
RETURN dis.id as disease_id, 
dis.label as disease_name,
collect(phe.label) as features,
count(nodes) as num_of_features
ORDER BY num_of_features DESC, disease_name
LIMIT 5
책 3.26: 0.0초, 5행

증상 조합에 맞는 질병 순위 (책 표 3.2 와 대조):
    5개  OMIM:619269    Ondontochondrodysplasia 2 with hearing loss and d

---
## 3.5 KG 위에서 추론하기

3.4절은 **명시적으로 저장된** 사실을 조회했다. 이 절은 **명시되지 않은** 사실을 논리로 끌어낸다 — 2장의 **연역적 추론(deductive reasoning)** 이다.

질문: **"내분비계의 이상으로 규정되는 질병은 무엇인가?"**

일부 주석은 "내분비계 이상"([`HP:0000818`](https://www.ebi.ac.uk/ols4/ontologies/hp/classes?obo_id=HP:0000818))에 직접 연결돼 있다. 하지만 임상의는 갑상선처럼 **더 구체적인** 표현형 특징에도 관심이 있다. 어떤 질병이 `Hypothyroidism`(갑상선 저하증)에 연결돼 있다면, 갑상선 저하증이 내분비계 이상의 하위 클래스이므로 **그 질병도 내분비계 이상과 연관된다** — 그래프에 그 간선이 없어도 참이다.

이 추론을 가능하게 하는 것이 **온톨로지에서 온 `SUBCLASSOF` 계층** 이다. 그림 3.8 오른쪽에서 본 그 계층이다. 3.3절에서 `hp.owl` 을 적재한 덕에 이것이 그래프 안에 있다. RDF의 의미론적 자산을 LPG로 끌어온 것이 여기서 값을 낸다.

- **책 3.27** 은 계층을 직접 훑는다. `[:SUBCLASSOF*1..3]` 은 "하위 클래스 관계를 1\~3단계까지 따라간다"는 가변 길이 경로 패턴이다

- **책 3.28** 은 [`n10s.inference.nodesInCategory`](https://neo4j.com/labs/neosemantics/4.3/inference/) 로 그 일을 프로시저에 맡긴다. `inCatRel` 은 질병을 표현형에 잇는 관계(`HAS_PHENOTYPIC_FEATURE`), `subCatRel` 은 하위 클래스 관계(`SUBCLASSOF`)를 지정한다


In [14]:
"""책 3.27 + 3.28 — 계층 순회와 온톨로지 기반 추론.

3.28 의 결과에서, 각 질병의 features 목록에 '내분비계 이상' 자체는 없지만
그 하위 클래스(Hypothyroidism, Goiter 등)가 들어 있다. 그래서 이 질병들이
'내분비계 이상' 카테고리에 암묵적으로 속한다고 추론된다.

주의: 책 3.27 은 예시로 깊이를 *1..3 으로 제한하지만,
n10s.inference.nodesInCategory 는 SUBCLASSOF 계층을 깊이 제한 없이 훑는다.
아래 ★ 판정은 프로시저와 같은 기준(무제한 깊이)을 쓴다.
"""
show("3.27")
rows = run("3.27", limit=0)
print(f"→ HP:0000818(내분비계 이상)의 1~3단계 하위 클래스 경로 {len(rows)}개")

with cypher.driver() as drv, drv.session(database=DB) as s:
    shallow = s.run("""MATCH (p:HpoPhenotype {id:"HP:0000818"})<-[:SUBCLASSOF*1..3]-(n)
                       RETURN count(DISTINCT n) AS n""").single()["n"]
    endocrine = {r["label"] for r in s.run(
        """MATCH (p:HpoPhenotype {id:"HP:0000818"})<-[:SUBCLASSOF*]-(n:HpoPhenotype)
           RETURN DISTINCT n.label AS label""").data()}
    print(f"   깊이 1~3 하위 클래스 {shallow}개 / 깊이 무제한 {len(endocrine)}개")
    print("   -> 깊이를 제한하면 'Transient neonatal diabetes mellitus'(깊이 4) 같은")
    print("      항목을 놓친다. 추론 프로시저를 쓰는 이유가 이것이다.")

print("\n" + "=" * 78)
show("3.28")
inferred = run("3.28", limit=0)

print("\n추론 결과 (책 표 3.3 과 대조) — ★ 는 '내분비계 이상'의 하위 클래스:")
for r in inferred:
    print(f"\n  {r['disease']}")
    for feature in sorted(r["features"]):
        print(f"     {'★' if feature in endocrine else ' '} {feature}")

print("\n※ 책 표 3.3 은 'Hyperglycemia' 도 굵게 표시하지만, 현재 HPO 에서 이 용어는")
print("  'Abnormality of the endocrine system' 하위가 아니다 (대사 이상 쪽으로 분류).")
print("  책 집필 시점 이후 온톨로지가 재구성된 결과다 — 추론 결과는 온톨로지 구조에")
print("  직접 의존하므로, 온톨로지가 바뀌면 추론도 바뀐다는 점을 보여준다.")

── 책 Listing 3.27  (파일 3.30) ──
MATCH (p:HpoPhenotype)<-[:SUBCLASSOF*1..3]-(n:HpoPhenotype)// #A
WHERE p.id = "HP:0000818"
RETURN p,n
책 3.27: 0.0초, 276행
→ HP:0000818(내분비계 이상)의 1~3단계 하위 클래스 경로 276개
   깊이 1~3 하위 클래스 256개 / 깊이 무제한 474개
   -> 깊이를 제한하면 'Transient neonatal diabetes mellitus'(깊이 4) 같은
      항목을 놓친다. 추론 프로시저를 쓰는 이유가 이것이다.

── 책 Listing 3.28  (파일 3.31) ──
MATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})
CALL n10s.inference.nodesInCategory(cat, { 
    inCatRel: "HAS_PHENOTYPIC_FEATURE", subCatRel: "SUBCLASSOF"})
YIELD node as dis
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label as disease, collect(DISTINCT phe.label) as features
ORDER BY size(features) ASC, disease
SKIP 100
LIMIT 5


책 3.28: 0.1초, 5행

추론 결과 (책 표 3.3 과 대조) — ★ 는 '내분비계 이상'의 하위 클래스:

  Congenital atransferrinemia
       Abnormality of the cardiovascular system
       Abnormality of the pancreas
       Anemia
       Arthritis
     ★ Hypothyroidism
       Recurrent infections

  Deafness, autosomal recessive 4, with enlarged vestibular aqueduct
       Autosomal recessive inheritance
       Congenital onset
       Enlarged vestibular aqueduct
     ★ Goiter
       Incomplete partition of the cochlea type II
       Sensorineural hearing impairment

  Diabetes mellitus, transient neonatal, 1
       Autosomal dominant inheritance
       Dehydration
       Hyperglycemia
       Intrauterine growth retardation
       Severe failure to thrive
     ★ Transient neonatal diabetes mellitus

  Edema, familial idiopathic, prepubertal
       Abnormality of the genitourinary system
       Autosomal dominant inheritance
     ★ Diabetes mellitus
       Edema
       Irritability
       Vomiting

  Familial dysalbuminemic h

---
## 실습 3-C — 해설판 3.4의 연습문제

> 리스팅 3.26의 질의를 확장하여, `evidence_name`, `evidence_description`, `source`, `url` 을 포함한 관계 속성들을 조회해 보세요.

이 연습은 3.2절의 기술 선택이 왜 옳았는지를 확인하는 마무리다. 진단 후보를 얻는 것으로 끝나지 않고, **각 연관의 근거를 임상의가 직접 열어볼 수 있어야** 한다는 3.1절의 요구사항으로 되돌아온다. 그림 3.6이 예고한 "간선에 붙은 속성"이 여기서 실제로 쓰인다.

> **주의** — 해설판 연습문제 지문은 `evidence_name` 처럼 snake_case로 적혀 있지만, 책 3.23이 실제로 만드는 속성명은 **camelCase** 다.
>
> `evidenceName`, `evidenceDescription`, `aspectName`, `aspectDescription`, `createdBy`, `creationDate`, `url`

In [15]:
"""실습 3-C 풀이 — 책 3.26 을 확장해 관계의 근거 속성까지 조회한다.

책 3.26 은 질병과 매칭 개수만 반환한다. 여기서는 매칭된 관계 하나하나의
출처·증거·큐레이터·열람 URL 까지 끌어와, 임상의가 근거를 추적할 수 있게 한다.
"""
PATIENT_FEATURES = [
    "Growth delay", "Large knee", "Sensorineural hearing impairment",
    "Pruritus", "Type I diabetes mellitus",
]

QUERY = """
MATCH (phe:HpoPhenotype) WHERE phe.label IN $features
MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe)
WITH dis, count(DISTINCT phe) AS matched,
     collect({feature: phe.label, evidence: rel.evidenceName,
              description: rel.evidenceDescription, source: rel.source,
              url: rel.url, curator: rel.createdBy, date: rel.creationDate}) AS grounds
RETURN dis.id AS disease_id, dis.label AS disease, matched, grounds
ORDER BY matched DESC, disease
LIMIT 2
"""

with cypher.driver() as drv, drv.session(database=DB) as s:
    for row in s.run(QUERY, features=PATIENT_FEATURES).data():
        print(f"\n{'=' * 78}\n{row['disease_id']}  {row['disease']}  "
              f"— 특징 {row['matched']}개 일치")
        for g in row["grounds"]:
            if g["feature"] not in PATIENT_FEATURES:
                continue
            print(f"\n  · {g['feature']}")
            print(f"      증거   : {g['evidence']}")
            print(f"      출처   : {g['source']}   ->  {g['url']}")
            print(f"      큐레이션: {g['curator']} ({g['date']})")
            if g["description"]:
                print(f"      설명   : {g['description'][:96]}...")

print("\n→ 진단 후보뿐 아니라 '왜 그렇게 판단했는지' 의 근거까지 관계에서 나온다.")
print("  RDF 로 이 정보를 관계에 붙이려면 N항 관계·명명된 그래프·RDF-star 중")
print("  하나로 우회해야 했다. 3.2절에서 LPG 를 고른 결과가 이것이다.")


OMIM:619269  Ondontochondrodysplasia 2 with hearing loss and diabetes  — 특징 5개 일치

  · Pruritus
      증거   : Published clinical study
      출처   : PMID:32101163   ->  https://pubmed.ncbi.nlm.nih.gov/32101163
      큐레이션: probinson (2021-06-20)
      설명   : PCS is used for information extracted from articles in the medical literature. Generally, annota...

  · Growth delay
      증거   : Published clinical study
      출처   : PMID:32101163   ->  https://pubmed.ncbi.nlm.nih.gov/32101163
      큐레이션: probinson (2021-06-20)
      설명   : PCS is used for information extracted from articles in the medical literature. Generally, annota...

  · Sensorineural hearing impairment
      증거   : Published clinical study
      출처   : PMID:32101163   ->  https://pubmed.ncbi.nlm.nih.gov/32101163
      큐레이션: probinson (2021-06-20)
      설명   : PCS is used for information extracted from articles in the medical literature. Generally, annota...

  · Large knee
      증거   : Published clinical study
      출처   : 

---
## 요약

- KG 구축은 **문제 정의 → 도메인 이해 → 데이터 이해 → 준비 → 모델 생성 → 질의** 의 과정이다 (그림 3.1·3.2). 코드보다 앞선 이해 단계가 기술 선택을 결정한다

- 결과물은 여러 소스를 **통일되고 근거가 탄탄하며 의미 있게** 표현해야 한다. 그 수단이 **온톨로지를 기준 스키마로 채택하는 의미 통합** 이다

- **RDF** 는 지식 표현과 온톨로지 구성에 강하고, **LPG** 는 질의 기반 순회와 **간선별 속성** 에 강하다. 이 장은 "주석마다 근거가 다르다"는 요구 때문에 LPG를 골랐고(그림 3.6), RDF의 계층 자산은 **Neosemantics** 로 끌어왔다

- 두 모델의 차이를 아는 것이 목적에 맞는 기술을 고르는 데 결정적이다

### 이 노트북에서 실측한 수치

| 항목 | 값 |
| --- | --- |
| 적재된 트리플 | 926,450 (책 집필 시점 899,558) |
| `HpoPhenotype` 노드 | 20,413 |
| `HpoDisease` 노드 | 12,956 |
| `HAS_PHENOTYPIC_FEATURE` 관계 | 284,994 |
| 정리 단계에서 삭제 | 207,754 노드 |
| 진단 질의 최상위 후보 | `OMIM:619269` Ondontochondrodysplasia 2 (특징 5개 일치) |

책 표 3.1·3.2·3.3 의 핵심 결과가 모두 재현되었다. 하위 순위와 절대 수치는 HPO 데이터가 계속 갱신되므로 책과 다를 수 있다.

### 핵심 용어

| 용어 | 뜻 |
| --- | --- |
| 온톨로지 (ontology) | 도메인의 명칭·속성·범주·관계를 표준 어휘로 정의한 참조 스키마 |
| 의미 통합 (semantic integration) | 표현이 달라도 같은 개념을 하나로 묶어 데이터를 통합하는 작업 |
| 매핑 (mapping) | 소스의 로컬 스키마를 온톨로지의 기준 스키마에 잇는 대응 |
| 트리플 (triple) | 주어·술어·목적어로 이루어진 RDF의 문장 단위 |
| OWL | RDF의 의미 정보를 풍부히 해 클래스·속성 정의를 지원하는 언어 |
| 공백 노드 (blank node) | 전역 식별자 없이 정보를 묶는 이름 없는 RDF 자원 |
| RDF-star | 트리플 자체에 속성을 붙일 수 있게 한 RDF 확장 |
| Neosemantics (n10s) | Neo4j에서 RDF/OWL을 다루고 기본 추론을 실행하는 플러그인 |
| 추론 (inference) | 명시되지 않은 정보를 논리 규칙으로 도출하는 연역 |
| 출처 추적 (provenance tracking) | 정보의 출처·작성자·날짜를 추적 가능하게 기록하는 것 |

### 참고 링크

- [교재 3장 해설판 (한국어)](https://github.com/restful3/ds4th_study/blob/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_03_create_your_first_knowledge_graph_from_ontologies/03_create_your_first_knowledge_graph_from_ontologies_ko_explained.md)

- [원서 저장소 chapters/ch03](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/tree/main/chapters/ch03)

- [Human Phenotype Ontology](https://hpo.jax.org/)

- [Neosemantics (n10s) 문서](https://neo4j.com/labs/neosemantics/)

### 다음 장으로

4장(저장소 [`ch04`](https://github.com/restful3/ds4th_study/tree/main/source/Alessandro%20Negro%20-%20Knowledge%20Graphs%20and%20LLMs%20in%20Action/chapter_04_from_simple_networks_to_multisource_integration/src/ch04))은 같은 생의학 도메인에서 **여러 소스를 아우르는 통합** 으로 나아간다. PPI 네트워크와 Het.io 를 다루고, 커뮤니티 탐지(Louvain·WCC) 같은 그래프 알고리즘과 LLM 을 결합해 결과를 해석한다. 3장이 "한 온톨로지로 짓기" 였다면 4장은 "여러 소스를 붙이기" 다.

주의: 4장의 리스팅 번호 오프셋은 **0** 이다 (책 4.2 = 파일 `4.2 Import PPI Network`). 3장의 −3 을 그대로 적용하면 안 된다.